In [2]:
#import
from datetime import datetime
from pydantic import BaseModel, PositiveInt, ValidationError

#Class 
class User(BaseModel):
    id : int
    name : str = 'John Doe'
    signup_ts : datetime | None
    tastes : dict[str, PositiveInt]
    
#Data
external_data = {
    'id' : 123,
    'signup_ts' : '2019-06-01 12:22',
    'tastes': {
        'wine' : 9,
        b'cheese' : 7,
        'cabbage' : '1',
    },
}

# Wrong Data
external_data_2 = {
    'id' : 'not an int',
    'tastes' : {}
}

if __name__ == "__main__":
    #  '**'을 붙이는 이유는 ditionay unpacking이다.
    #  class 초기화시에 딕셔너리 값을 통째로 넣어주는 것보다 언패킹하여서 Class와 형태를 맞춰줌. 
    user = User(**external_data)
    print(user.id)
    #> 123
    
    print(user.model_dump())
    """
    {
        'id' : 123,
        'name' : 'John Doe',
        'signup_ts' : datetime.datetime(2019, 6, 1, 12, 22),
        'tastes' : {'wine' : 9, 'cheese' : 7, 'cabbage' : 1}, 
    
    }
    """
    print(type(user.tastes['cabbage']))
    #type 을 변경해줌.
   
    try:
       User(**external_data_2)
    except ValidationError as e:
        print(e.errors())
    except:
        print("Validation Error가 아닌 경우 다음 exeption 탐색")
    else:
        print("exception들이 실행되지 않으면 즉 try구문이 정상실행되면 해당 구간 실행")
    finally:
        print("항상 마지막에 실행 예를 들어 파일 시스템 실행시 마지막에 f.close()로 사용")
        
    """
    [
        {
            'type': 'int_parsing', 
            'loc': ('id',), 
            'msg': 'Input should be a valid integer, unable to parse string as an integer', 
            'input': 'not an int', 
            'url': 'https://errors.pydantic.dev/2.13/v/int_parsing'
        }, 
        {
            'type': 'missing', 
            'loc': ('signup_ts',), 
            'msg': 'Field required', 
            'input': {'id': 'not an int', 'tastes': {}}, 
            'url': 'https://errors.pydantic.dev/2.13/v/missing'
        }
    ]
    """

123
{'id': 123, 'name': 'John Doe', 'signup_ts': datetime.datetime(2019, 6, 1, 12, 22), 'tastes': {'wine': 9, 'cheese': 7, 'cabbage': 1}}
<class 'int'>
[{'type': 'int_parsing', 'loc': ('id',), 'msg': 'Input should be a valid integer, unable to parse string as an integer', 'input': 'not an int', 'url': 'https://errors.pydantic.dev/2.13/v/int_parsing'}, {'type': 'missing', 'loc': ('signup_ts',), 'msg': 'Field required', 'input': {'id': 'not an int', 'tastes': {}}, 'url': 'https://errors.pydantic.dev/2.13/v/missing'}]
항상 마지막에 실행 예를 들어 파일 시스템 실행시 마지막에 f.close()로 사용


In [3]:
from typing import Annotated, Literal
from annotated_types import Gt
from pydantic import BaseModel
from datetime import datetime

# Annotated는 추가 값을 검증하기 위한 기능
# Gt(x) 는 Grater than 즉 ()안의 매개 변수 x 보다 값이 커야한다는 의미이다.
#Literal의 경우 [ ]안에 있는 'red' 혹은 'green' 만 가능하다.

class Fruit(BaseModel):
    name: str
    color: Literal['red', 'green']
    weight: Annotated[float, Gt(0)]
    bazam: dict[str, list[tuple[int, bool, float]]]
def testTypeHint():
    print(
        Fruit(
            name= 'Apple', 
            color= 'red',
            weight= 4.2,
            bazam= {'foobar': [(1, True, 0.1)]},
        )    
    )
    
class Meeting(BaseModel):
    when: datetime
    where: bytes
    why: str = 'No idea'
    
    
#serialization 직렬화는 pydantic 객체를 dictionary 또는 json화 하는 것
#model_dump를 통해 직렬화되고 안에 매개변수 옵션 설정이 가능하다.

def test_serialization_3_ways():
    m = Meeting(when= '2020-01-01T12:00', where= 'home')
    
    print(m.model_dump(exclude_unset=True))
    #{'when': datetime.datetime(2020, 1, 1, 12, 0), 'where': b'home'}
    print(m.model_dump(exclude= {'where'}, mode= 'json'))
    print(type(m.model_dump(exclude= {'where'}, mode= 'json')))
    #{'when': '2020-01-01T12:00:00', 'why': 'No idea'}
    #반환 타입이 Dictionary
    print(m.model_dump_json(exclude_defaults=True))
    print(type(m.model_dump_json(exclude_defaults=True)))
    #{"when":"2020-01-01T12:00:00","where":"home"
    #반환 타입이 String

if __name__ == "__main__":
    # testTypeHint()
    test_serialization_3_ways()
    

{'when': datetime.datetime(2020, 1, 1, 12, 0), 'where': b'home'}
{'when': '2020-01-01T12:00:00', 'why': 'No idea'}
<class 'dict'>
{"when":"2020-01-01T12:00:00","where":"home"}
<class 'str'>
